In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv(r"netflix_titles_cleaned.csv")

In [ ]:
X = df[['release_year', 'duration']].dropna()

X

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

inertia = []
K_range = range(1,10)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42,n_init=10)
    kmeans.fit(X)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8,4))
plt.plot(K_range, inertia,marker='o',linestyle='--', color='b')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Elbow method for optimal k')
plt.grid(True)
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=3,random_state=42,n_init=10)
df['cluster'] = kmeans.fit_predict(X)

print(df['cluster'].value_counts())

In [ ]:
cluster_summary = df.groupby('cluster')[['release_year', 'duration']].mean()
print(cluster_summary)

In [ ]:
plt.figure(figsize=(10,6))
scatter = plt.scatter(df['release_year'],df['duration'],c=df['cluster'],cmap='viridis',alpha=0.6)
plt.colorbar(scatter,label='cluster id')
plt.xlabel('release year')
plt.ylabel('duration')
plt.grid(True)
plt.show()

In [ ]:
cluster_name = {
    0: 'Modern Movies',
    1: 'Modern Shows / Short Content',
    2: 'Classic & Vintage Catalog'
}

df['cluster_name'] = df['cluster'].map(cluster_name)

df[['title','type','release_year','duration','cluster_name']].head(10)

## part 2: buliding a distance-based content recommendation engine 


In [ ]:
from scipy.spatial.distance import cdist

def recommend_content(title_name,df,feature_scaled,top_n=5):
    matches = df[df['title'].str.lower() == title_name.lower()]

    if matches.empty:
        return f"title '{title_name}' not found in the dataset try another one"

    target_idx = matches.index[0]
    target_cluster = df.loc[target_idx,'cluster_name']

    target_vector = feature_scaled[target_idx].reshape(1,-1)

    distances = cdist(feature_scaled,target_vector, metric='euclidean').flatten()

    df_temp = df.copy()
    df_temp['distence'] = distances

    recommendations = df_temp[df_temp.index != target_idx].sort_values(by='distence')

    print(f"--- Top {top_n} Recommendations for '{df.loc[target_idx,'title']}' ---")
    print(f"Category: {target_cluster}  Year: {df.loc[target_idx,'release_year']}  Duration: {df.loc[target_idx, 'duration']}\n")

    return recommendations[['title','type','release_year','duration','cluster_name','distence']].head(top_n)

In [ ]:
recommend_content('Blue Mountain State: The Rise of Thadland',df,X)

In [ ]:
recommend_content('3%',df,X)

* **Unsupervised Clustering:** Applied K-Means clustering with `StandardScaler` on `release_year` and `duration_num` to segment the dataset into 3 natural groups (Modern Movies, Modern TV Shows, Classic Catalog) determined by the Elbow Method.
* **Vector Recommendations:** Engineered a Euclidean distance content recommender using `scipy.spatial.distance.cdist` to find nearest-neighbor titles in feature space.
* **Key Insight:** Feature scaling is mandatory for distance-based algorithms like K-Means and KNN to prevent large-scale features from dominating distance metrics.